# MirrorTopology T2b-2 符号地図 v0.6（本番用：NotebookIdentity最終確認レビュー反映）
2026-08-26。判定規則v0.3準拠。v0.5からの変更（唯一の残余BLOCKER＋推奨2点）：

1. **production時のnotebook identity照合を必須化**（§5–8）：production modeでは
   `NOTEBOOK_PATH`（Drive clone内の実行中ファイル）の指定・存在・
   `REPO_GATE['nb_local_match'] is True` を**hard assert**。「pathを渡さなければ照合が
   スキップされる」抜け道を閉鎖。
2. **public preregistration検証**（§11）：ゲートが `origin` のURL（正準URLと正規化比較）と
   **HEADコミットのpush済み確認**（read-onlyの`git ls-remote`）を記録し，productionでは
   両方をassert。証拠鎖＝ public remote commit → clone → exact HEAD verification → production。
3. **V7 cross-check手順の事前凍結**（§13）：candidate向けV7検証を関数
   `t2b2_run.v7_crosscheck`としてコミット（既存V7実空間検証を候補(幾何,セクター)へ
   そのまま適用・SW・ℓ≤30・|ratio−1|<1e-2）。**候補を見る前に方法を固定**。

（参考）source-only SHA実装はレビュー側と完全一致確認済み（v0.5で10ce7675…を双方が算出）。

In [ ]:
import os, sys, hashlib, json
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        raise RuntimeError('Driveマウント失敗：ここで明示停止します。詳細: ' + repr(e))
    BASE = '/content/drive/MyDrive/mirror_topology'
    if not os.path.isdir(BASE):
        raise RuntimeError('mirror_topology フォルダが見つかりません: ' + BASE)
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'camb', 'healpy'], check=True)
else:
    BASE = '/home/claude/colab_sim'
    os.makedirs(BASE, exist_ok=True)
OUT = os.path.join(BASE, 'runs_t2b2')
os.makedirs(OUT, exist_ok=True)
BPM_ARRAY_SHA256 = '9693b2072f3dc3e63540733b3d74b70fda2541d17cc2e60d34b6cb71311ae8a0'   # 規則v0.3凍結：sha256(Bp||Bm)（source of truth）
BPM_FILE_SHA256_NOTE = '5bf6e5167995515c5f0a38f4e126ca252bbf8a28e7e6c90fe0d167481871e0f0'  # 参考：npzコンテナのhash（再保存で変わり得る・副次記録）
BPM_CANDS = ([os.path.join(BASE, 'step0_frozen_Bpm_v1.npz'),
              os.path.join(BASE, 'runs_step0', 'step0_frozen_Bpm_v1.npz')] if IN_COLAB
             else ['/home/claude/step0_frozen_Bpm.npz'])
BPM_PATH = next((p for p in BPM_CANDS if os.path.exists(p)), None)
if BPM_PATH is None and IN_COLAB:
    import subprocess
    for br_ in ['main', 'master']:
        url = f'https://raw.githubusercontent.com/tsujikeita/mirror-topology/{br_}/step0_frozen_Bpm_v1.npz'
        r = subprocess.run(['curl', '-sfL', '-o', os.path.join(BASE, 'step0_frozen_Bpm_v1.npz'), url])
        if r.returncode == 0:
            BPM_PATH = os.path.join(BASE, 'step0_frozen_Bpm_v1.npz'); break
assert BPM_PATH and os.path.exists(BPM_PATH), 'step0_frozen_Bpm_v1.npz が見つかりません'
import numpy as np
_z = np.load(BPM_PATH, allow_pickle=True)
_arr_sha = hashlib.sha256(_z['Bp'].tobytes() + _z['Bm'].tobytes()).hexdigest()
assert _arr_sha == BPM_ARRAY_SHA256, f'B± 科学配列SHA256不一致: {_arr_sha}'
BPM_FILE_SHA = hashlib.sha256(open(BPM_PATH, 'rb').read()).hexdigest()
print('OUT =', OUT, '/ B± array-sha OK (file sha:', BPM_FILE_SHA[:12], '…)')

In [ ]:
%%writefile t2b2_core.py
"""T2b-2 core v0.3: E7 twisted eigenmodes (full Bieberbach BC), covariances, quadratic sky projection.
Units: lengths in chi_* (comoving radius of LSS) = 1.
v0.2 changes (2026-08-24, session 2):
  - integer lattice coordinates for all wavevectors (exact dedup keys, no float rounding)
  - memory-lean two-pass pair engine sized for production grids
  - sky_cov_q_multi: several transfer functions share one mode/pair/unique-K construction
  - re-verified: V1/V3/V4/V5/V7 battery + regression against session-1 CSV
v0.3 change (2026-08-26): composite K=0 modes explicitly zeroed for l>0 (full-transfer interp
  would otherwise clamp to the q-grid minimum; SW was safe since j_l(0)=0 for l>0).

E7 (rectangular lattice) generators:
  g_A: x -> M_A x + T_A,  M_A = diag(1,-1,1), T_A = (LAx, LAy, 0);  g_A^2 = t_(2LAx,0,0)
  t1 : y -> y + L1y ; t2 : z -> z + L2z ;  H1 = Z(g_A) + Z(t2) + Z2(t1)
Real characters sigma=(sA,s1,s2). Twisted dual lattice:
  kx = pi*nx/LAx ; ky = 2pi(my+ay)/L1y, ay=(1-s1)/4 ; kz = 2pi(mz+az)/L2z, az=(1-s2)/4.
ky>0 orbit-pair: psi = [e^{ik.x} + sA e^{-i(Mk).T_A} e^{i(Mk).x}]/sqrt2 (e^{2i kx LAx}=1 auto).
ky=0 (s1=+1 only): single wave, parity selection (-1)^nx = sA.
P(k) = k^-3 (scale-invariant baseline; amplitude irrelevant for signs and rho).
Integer coords: mode wave = ints (nx, my, mz) with ky = sy*(my + twy/2), twy=(1-s1)/2 in {0,1}.
Composite (quadratic) wave = ints summed; K_y = sy*(iy + twy) etc. (untwisted lattice).
"""
import numpy as np
from scipy.special import spherical_jn
try:
    from scipy.special import sph_harm_y
    def Ylm(l, m, theta_pol, phi_az):
        return sph_harm_y(l, m, theta_pol, phi_az)
except ImportError:
    from scipy.special import sph_harm
    def Ylm(l, m, theta_pol, phi_az):
        return sph_harm(m, l, phi_az, theta_pol)

TOL = 1e-9

class E7Twisted:
    def __init__(self, LAx, L1y, L2z, LAy, sA, s1, s2, kcut):
        self.p = dict(LAx=LAx, L1y=L1y, L2z=L2z, LAy=LAy, sA=sA, s1=s1, s2=s2, kcut=kcut)
        self.TA = np.array([LAx, LAy, 0.0])
        self.twy = 0 if s1 == 1 else 1
        self.twz = 0 if s2 == 1 else 1
        self.sx = np.pi / LAx; self.sy = 2 * np.pi / L1y; self.sz = 2 * np.pi / L2z
        self._build_reps()
        self._realify()

    # ---- integer <-> float wavevectors (mode lattice: y-shift twy/2, z-shift twz/2) ----
    def _kfloat(self, iQ):
        K = np.empty(np.shape(iQ), float)
        iQ = np.asarray(iQ)
        K[..., 0] = self.sx * iQ[..., 0]
        K[..., 1] = self.sy * (iQ[..., 1] + 0.5 * self.twy)
        K[..., 2] = self.sz * (iQ[..., 2] + 0.5 * self.twz)
        return K

    def _ineg(self, t):    # integer rep of -k on the mode lattice
        return (-t[0], -t[1] - self.twy, -t[2] - self.twz)

    def _imir(self, t):    # integer rep of M_A k (flip y)
        return (t[0], -t[1] - self.twy, t[2])

    # ---------- twisted-mode construction ----------
    def _build_reps(self):
        kcut, sA = self.p['kcut'], self.p['sA']
        nxm = int(np.floor(kcut / self.sx)) + 1
        nym = int(np.floor(kcut / self.sy)) + 2
        nzm = int(np.floor(kcut / self.sz)) + 2
        reps = {}
        for nx in range(-nxm, nxm + 1):
            kx = self.sx * nx
            for my in range(-nym, nym + 1):
                ky = self.sy * (my + 0.5 * self.twy)
                if ky < -TOL:
                    continue                      # orbit representative: ky>0 or ky==0
                for mz in range(-nzm, nzm + 1):
                    kz = self.sz * (mz + 0.5 * self.twz)
                    kn = np.sqrt(kx * kx + ky * ky + kz * kz)
                    if kn > kcut or kn < TOL:
                        continue
                    t = (nx, my, mz)
                    if abs(ky) < TOL:             # single-wave sector (needs s1=+1)
                        if (-1) ** nx != sA:
                            continue              # parity selection e^{i kx LAx} = sA
                        reps[t] = dict(waves=[(t, 1.0 + 0j)], kmag=kn, kind='s')
                    else:                         # orbit pair {k, M_A k}
                        tm = self._imir(t)
                        Mk = self._kfloat(np.array(tm, float)[None, :])[0]
                        c2 = sA * np.exp(-1j * (Mk @ self.TA))
                        reps[t] = dict(waves=[(t, 1 / np.sqrt(2) + 0j), (tm, c2 / np.sqrt(2))],
                                       kmag=kn, kind='p')
        self.reps = reps

    def _closed(self, waves, coef_map):
        """conjugate-closed dict {int3: coeff} for real combo a*psi + b*conj(psi)."""
        a, b = coef_map
        d = {}
        for t, c in waves:
            for tt, cc in ((t, a * c), (self._ineg(t), b * np.conj(c))):
                d[tt] = d.get(tt, 0j) + cc
        return d

    def _realify(self):
        consumed, modes = set(), []
        for key, rep in self.reps.items():
            if key in consumed:
                continue
            nx, my, mz = key
            pkey = self._imir(self._ineg(key))    # conjugate orbit representative (ky>=0)
            if pkey == key:
                # self-conjugate orbit spans ONE real dimension when psi* prop psi:
                # Gram-Schmidt over Re/Im candidates keeps only independent ones.
                kept = []
                for cm in [(0.5, 0.5), (-0.5j, 0.5j)]:
                    d = self._closed(rep['waves'], cm)
                    for kd in kept:
                        ip = sum(v * np.conj(kd[q]) for q, v in d.items() if q in kd)
                        for q in d:
                            if q in kd:
                                d[q] -= ip * kd[q]
                    n2 = sum(abs(v) ** 2 for v in d.values())
                    if n2 > 1e-10:
                        s = 1 / np.sqrt(n2)
                        dn = {q: v * s for q, v in d.items()}
                        kept.append(dn)
                        modes.append(dict(d=dn, kmag=rep['kmag']))
                consumed.add(key)
            else:
                assert pkey in self.reps, f"conjugate partner missing for {key}"
                for cm in [(1 / np.sqrt(2), 1 / np.sqrt(2)), (-1j / np.sqrt(2), 1j / np.sqrt(2))]:
                    d = self._closed(rep['waves'], cm)
                    n2 = sum(abs(v) ** 2 for v in d.values())
                    s = 1 / np.sqrt(n2)
                    modes.append(dict(d={q: v * s for q, v in d.items()}, kmag=rep['kmag']))
                consumed.add(key); consumed.add(pkey)
        W = max(len(m['d']) for m in modes)
        N = len(modes)
        iQ = np.zeros((N, W, 3), np.int64); D = np.zeros((N, W), complex); nw = np.zeros(N, int)
        for i, m in enumerate(modes):
            for j, (t, v) in enumerate(m['d'].items()):
                iQ[i, j] = t; D[i, j] = v
            nw[i] = len(m['d'])
        self.iQ, self.D, self.nwav = iQ, D, nw
        self.Q = self._kfloat(iQ)                 # floats derived exactly from ints
        self.kmag = np.array([m['kmag'] for m in modes])
        self.P = self.kmag ** -3.0
        self.N = N

    # ---------- evaluations ----------
    def eval_modes(self, X):
        ph = np.exp(1j * np.einsum('pc,nwc->pnw', X, self.Q))
        U = np.einsum('pnw,nw->pn', ph, self.D)
        assert np.abs(U.imag).max() < 1e-9 * max(1.0, np.abs(U.real).max())
        return U.real

    def Cphi_modes(self, X, Y):
        UX, UY = self.eval_modes(X), self.eval_modes(Y)
        return np.einsum('pn,qn,n->pq', UX, UY, self.P)

    def Cphi_pairs(self, X, Y):
        UX, UY = self.eval_modes(X), self.eval_modes(Y)
        return np.einsum('pn,pn,n->p', UX, UY, self.P)

    def Cphi_image(self, X, Y):
        """independent construction: direct + glide-image sum over the FULL shifted lattice."""
        p = self.p; sA = p['sA']; kcut = p['kcut']
        nxm = int(np.floor(kcut / self.sx)) + 1
        nym = int(np.floor(kcut / self.sy)) + 2
        nzm = int(np.floor(kcut / self.sz)) + 2
        ks = []
        for nx in range(-nxm, nxm + 1):
            for my in range(-nym, nym + 1):
                for mz in range(-nzm, nzm + 1):
                    k = np.array([self.sx * nx, self.sy * (my + 0.5 * self.twy),
                                  self.sz * (mz + 0.5 * self.twz)])
                    kn = np.linalg.norm(k)
                    if TOL < kn <= kcut:
                        ks.append(k)
        K = np.array(ks); P = np.linalg.norm(K, axis=1) ** -3.0
        MK = K * np.array([1.0, -1.0, 1.0])
        phg = np.exp(-1j * MK @ self.TA)
        E_x_k = np.exp(1j * X @ K.T); E_y_k = np.exp(-1j * Y @ K.T)
        E_x_Mk = np.exp(1j * X @ MK.T)
        direct = np.einsum('pk,qk,k->pq', E_x_k, E_y_k, P)
        image = np.einsum('pk,qk,k->pq', E_x_Mk * phg[None, :], E_y_k, P) * sA
        C = 0.5 * (direct + image)
        assert np.abs(C.imag).max() < 1e-8 * max(1.0, np.abs(C.real).max())
        return C.real

    def var_profile(self, ys):
        X = np.zeros((len(ys), 3)); X[:, 1] = ys
        return self.Cphi_pairs(X, X)

    def deck_check(self, nrand=40, seed=0):
        rng = np.random.default_rng(seed)
        p = self.p
        MA = np.array([1.0, -1.0, 1.0])
        def gA(x): return MA * x + self.TA
        def t1(x): return x + np.array([0, p['L1y'], 0])
        def t2(x): return x + np.array([0, 0, p['L2z']])
        words = [([gA], p['sA']), ([t1], p['s1']), ([t2], p['s2']),
                 ([gA, gA], 1), ([gA, t1], p['sA'] * p['s1']),
                 ([t1, t2, gA], p['s1'] * p['s2'] * p['sA']),
                 ([gA, t2, gA, t1], p['s1'] * p['s2'])]
        X = rng.uniform(-2, 2, size=(nrand, 3))
        worst = 0.0
        for maps, chi in words:
            GX = X.copy()
            for mp in maps[::-1]:
                GX = np.array([mp(x) for x in GX])
            U, GU = self.eval_modes(X), self.eval_modes(GX)
            worst = max(worst, np.abs(GU - chi * U).max())
        return worst

    def gram(self):
        G = np.zeros((self.N, self.N))
        dicts = []
        for i in range(self.N):
            dicts.append({tuple(self.iQ[i, j]): self.D[i, j] for j in range(self.nwav[i])})
        for i in range(self.N):
            for j in range(i, self.N):
                s = 0j
                for q, v in dicts[i].items():
                    mq = self._ineg(q)
                    if mq in dicts[j]:
                        s += v * dicts[j][mq]
                G[i, j] = G[j, i] = s.real
        return G

    def sample(self, X, nsamp, rng):
        U = self.eval_modes(X)
        g = rng.standard_normal((self.N, nsamp))
        return U @ (np.sqrt(self.P)[:, None] * g)

    # ---------- quadratic field -> sky (multi-transfer engine) ----------
    def sky_cov_q_multi(self, Fls, ls, chunk=4000):
        """C^{T,q}[(lm),(l'm')] for each transfer in Fls, sharing one construction.
        C = 2 sum_{n<=m} w P_n P_m S[u_n u_m] (x) S*[u_n u_m];  S via plane-wave products.
        Composite K=0 waves contribute only to l=0 (Kn_safe trick keeps that exact)."""
        lm = [(l, m) for l in ls for m in range(-l, l + 1)]
        nlm = len(lm)
        iu, ju = np.triu_indices(self.N)
        w = np.where(iu == ju, 1.0, 2.0)
        PPw = 2.0 * w * self.P[iu] * self.P[ju]
        Wp = self.iQ.shape[1]; W2 = Wp * Wp; NP = len(iu)
        B = np.int64(8192); OFF = np.int64(4096)
        # pass 1: packed integer keys of all composite wavevectors
        packs = np.empty(NP * W2, np.int64)
        for s0 in range(0, NP, chunk):
            sl = slice(s0, min(s0 + chunk, NP))
            iK = (self.iQ[iu[sl]][:, :, None, :] + self.iQ[ju[sl]][:, None, :, :]).reshape(-1, 3)
            packs[sl.start * W2: sl.stop * W2] = ((iK[:, 0] + OFF)
                                                  + (iK[:, 1] + OFF) * B
                                                  + (iK[:, 2] + OFF) * B * B)
        uniq, inv = np.unique(packs, return_inverse=True)
        del packs
        iz = uniq // (B * B) - OFF
        r = uniq % (B * B)
        iy = r // B - OFF
        ix = r % B - OFF
        # composite (untwisted) lattice: shifts double to integers twy, twz
        Kx = self.sx * ix.astype(float)
        Ky = self.sy * (iy.astype(float) + self.twy)
        Kz = self.sz * (iz.astype(float) + self.twz)
        Kn = np.sqrt(Kx * Kx + Ky * Ky + Kz * Kz)
        Kn_safe = np.maximum(Kn, 1e-12)
        th = np.arccos(np.clip(Kz / Kn_safe, -1, 1))
        ph = np.arctan2(Ky, Kx)
        zK = (Kn == 0)   # v0.3: composite K=0 contributes only to l=0; enforce exactly
        Ftabs = []
        for Fl in Fls:
            Ftab = np.zeros((nlm, len(uniq)), complex)
            for a, (l, m) in enumerate(lm):
                Ftab[a] = 4 * np.pi * (1j ** l) * Fl(l, Kn_safe) * np.conj(Ylm(l, m, th, ph))
                if l > 0:
                    Ftab[a][zK] = 0.0
            Ftabs.append(Ftab)
        Cs = [np.zeros((nlm, nlm), complex) for _ in Fls]
        # pass 2: gather + contract per chunk (cc rebuilt on the fly; memory-lean)
        for s0 in range(0, NP, chunk):
            sl = slice(s0, min(s0 + chunk, NP))
            nc = sl.stop - sl.start
            cc = (self.D[iu[sl]][:, :, None] * self.D[ju[sl]][:, None, :]).reshape(nc, W2)
            gi = inv[sl.start * W2: sl.stop * W2]
            for f, Ftab in enumerate(Ftabs):
                F_g = Ftab[:, gi].reshape(nlm, nc, W2)
                S = np.einsum('apw,pw->ap', F_g, cc)
                Cs[f] += (S * PPw[sl][None, :]) @ S.conj().T
        return Cs, lm

    def sky_cov_q(self, Fl, ls, chunk=4000, verbose=False):
        Cs, lm = self.sky_cov_q_multi([Fl], ls, chunk=chunk)
        return Cs[0], lm

    def A_refl_y(self, C, lm, ls):
        idx = {t: a for a, t in enumerate(lm)}
        out = {}
        for l in ls:
            s = 0j
            for m in range(-l, l + 1):
                s += (-1) ** m * C[idx[(l, m)], idx[(l, -m)]]
            out[l] = s.real / (4 * np.pi)
        return out

def F_SW(l, K, chi=1.0):
    return spherical_jn(l, K * chi) / 5.0

def band_stats(M, C, lm, band=(2, 3, 4)):
    """A_l per l, band sum, trace, rho for one covariance block."""
    A = M.A_refl_y(C, lm, list(band))
    idx = {t: a for a, t in enumerate(lm)}
    Tr = sum(C[idx[(l, m)], idx[(l, m)]].real for l in band for m in range(-l, l + 1))
    Ab = sum(A[l] for l in band)
    return A, Ab, Tr, 4 * np.pi * Ab / Tr

def extrapolate_k3(kcuts, vals):
    """least-squares fit v(k) = v_inf + c*k^-3; returns (v_inf, c, rms)."""
    x = np.asarray(kcuts, float) ** -3.0
    y = np.asarray(vals, float)
    Amat = np.stack([np.ones_like(x), x], 1)
    coef, *_ = np.linalg.lstsq(Amat, y, rcond=None)
    rms = float(np.sqrt(np.mean((Amat @ coef - y) ** 2)))
    return float(coef[0]), float(coef[1]), rms


In [ ]:
%%writefile t2b2_bridge.py
"""t2b2_bridge v1.2 (2026-08-26c): complex full-m C_q -> frozen-statistic (s+_q, s-_q).

Rules reference: T2b2_decision_rules_frozen_v0.3 Sec.3.1 (six-test battery) + review guards.

Conventions (frozen):
- Complex side: lm_full = [(l,m) for l in (2,3,4) for m in -l..l], C = <a a^H> (Hermitian PSD),
  physical real field => reality condition a_{l,-m} = (-1)^m a_{lm}^*.
- Real side: basis_lm = [(l,0,'c'), (l,1,'c'), (l,1,'s'), ...] matching step0_frozen_Bpm npz;
  x_{l0}=a_{l0}, x_c=(a_{lm}+(-1)^m a_{l,-m})/sqrt2, x_s=i(a_{lm}-(-1)^m a_{l,-m})/sqrt2.
- Alignment (implementation-level freeze, to be ratified in notebook review): the body-frame
  mirror normal y^ is rotated onto the frozen axis d^ = pix2vec(16,1134,ring); the azimuth psi
  about d^ is not fixed by the alignment hypothesis and the common mask breaks the symmetry, so
  s+-_q are reported as the average over N_PSI equally spaced psi with min/max recorded.
- Artifact identity: the scientific-array hash sha256(Bp||Bm) is the source of truth
  (rules v0.3 frozen value 9693b207...); the NPZ file hash is container-dependent and recorded
  as secondary provenance only.
- MC meaning: Gaussian covariance-MC validates the implementation of quadratic MEANS only; the
  physical q-field is non-Gaussian (distribution-level modelling deferred to Step 1+).
"""
import numpy as np
import healpy as hp

LS = (2, 3, 4)
N_PSI = 16          # l<=4: Fourier components up to 8 < 16, so 16-point uniform average is EXACT
RHO_NUM = 1e-6          # absolute numerical floor for rho classification
NOSIG_REL = 1e-9        # no-signal guard: s_tot < NOSIG_REL * max grid s_tot
SMALL_SM_REL = 1e-6     # numerical-small flag for g2max denominators

try:
    from scipy.special import sph_harm_y
    def _Ylm(l, m, th, ph): return sph_harm_y(l, m, th, ph)
except ImportError:
    from scipy.special import sph_harm
    def _Ylm(l, m, th, ph): return sph_harm(m, l, ph, th)


def lm_full(ls=LS):
    return [(l, m) for l in ls for m in range(-l, l + 1)]


def real_basis_lm(ls=LS):
    out = []
    for l in ls:
        out.append((l, 0, 'c'))
        for m in range(1, l + 1):
            out.append((l, m, 'c')); out.append((l, m, 's'))
    return out


def M_matrix(ls=LS):
    """x = M a  (rows: real basis, cols: full-m complex basis)."""
    lmf = lm_full(ls); rb = real_basis_lm(ls)
    idx = {t: a for a, t in enumerate(lmf)}
    M = np.zeros((len(rb), len(lmf)), complex)
    for i, (l, m, cs) in enumerate(rb):
        if m == 0:
            M[i, idx[(l, 0)]] = 1.0
        elif cs == 'c':
            M[i, idx[(l, m)]] = 1 / np.sqrt(2)
            M[i, idx[(l, -m)]] = ((-1) ** m) / np.sqrt(2)
        else:
            M[i, idx[(l, m)]] = 1j / np.sqrt(2)
            M[i, idx[(l, -m)]] = -1j * ((-1) ** m) / np.sqrt(2)
    return M, lmf, rb


def check_reality(C, ls=LS, rtol=1e-10):
    lmf = lm_full(ls); idx = {t: a for a, t in enumerate(lmf)}
    sc = np.abs(C).max(); worst = 0.0
    for (l, m) in lmf:
        for (l2, m2) in lmf:
            d = C[idx[(l, -m)], idx[(l2, -m2)]] - ((-1) ** (m + m2)) * np.conj(C[idx[(l, m)], idx[(l2, m2)]])
            worst = max(worst, abs(d) / sc)
    return worst, worst < rtol


def to_real(C, ls=LS, rtol=1e-10):
    M, lmf, rb = M_matrix(ls)
    Cr = M @ C @ M.conj().T
    im = float(np.abs(Cr.imag).max() / max(np.abs(Cr).max(), 1e-300))
    Cr = Cr.real
    asym = float(np.abs(Cr - Cr.T).max() / max(np.abs(Cr).max(), 1e-300))
    return Cr, dict(imag_rel=im, asym_rel=asym, ok=(im < rtol and asym < rtol))


def twopoint_check(C, Cr, ls=LS, npairs=50, seed=0, rtol=1e-10):
    """Independent mode-placement test: two-point function agreement complex vs real side."""
    rng = np.random.default_rng(seed)
    lmf = lm_full(ls); rb = real_basis_lm(ls)
    th = np.arccos(rng.uniform(-1, 1, 2 * npairs)); ph = rng.uniform(0, 2 * np.pi, 2 * npairs)
    Yc = np.array([[_Ylm(l, m, t, p) for (t, p) in zip(th, ph)] for (l, m) in lmf])   # (21, 2n)
    # real basis functions e_i(n) evaluated at the same points
    E = np.zeros((len(rb), 2 * npairs))
    for i, (l, m, cs) in enumerate(rb):
        Y = np.array([_Ylm(l, m, t, p) for (t, p) in zip(th, ph)])
        E[i] = (Y.real if m == 0 else (np.sqrt(2) * Y.real if cs == 'c' else np.sqrt(2) * Y.imag))
    worst = 0.0
    for k in range(npairs):
        a, b = k, npairs + k
        tc = float(np.real(Yc[:, a].conj() @ C.T @ Yc[:, b]))   # sum_ab C_ab Y_a(n) Y_b*(n') -> real part
        tc2 = float(np.real(np.einsum('a,ab,b->', Yc[:, a], C, np.conj(Yc[:, b]))))
        tr_ = float(E[:, a] @ Cr @ E[:, b])
        sc = max(abs(tc2), np.abs(Cr).max())
        worst = max(worst, abs(tc2 - tr_) / sc)
    return worst, worst < rtol


def frozen_axis_vec():
    return np.array(hp.pix2vec(16, 1134))


def rotation_frames(n_psi=N_PSI):
    d = frozen_axis_vec()
    ref = np.array([0.0, 0.0, 1.0])
    e1_0 = np.cross(ref, d); e1_0 /= np.linalg.norm(e1_0)
    e3_0 = np.cross(e1_0, d)
    R3s = []
    for k in range(n_psi):
        psi = 2 * np.pi * k / n_psi
        e1 = np.cos(psi) * e1_0 + np.sin(psi) * e3_0
        e3 = np.cross(e1, d)
        R3s.append(np.stack([e1, d, e3], axis=1))   # columns: images of x^_body, y^_body, z^_body
    return R3s


def rotation_O(R3, nside_fit=32, lmax_fit=8, tol=3e-6):
    """Orthogonal 21x21: coefficients of rotated real-basis fields. Self-validating."""
    rb = real_basis_lm()
    npix = hp.nside2npix(nside_fit)
    V = np.array(hp.pix2vec(nside_fit, np.arange(npix))).T
    Vb = V @ R3                      # body-frame coordinates of pixel directions (R3^{-1} n = R3^T n)
    thb, phb = hp.vec2ang(Vb)
    O = np.zeros((len(rb), len(rb)))
    almL = {}
    for j, (l, m, cs) in enumerate(rb):
        Y = _Ylm(l, m, thb, phb)
        T = (Y.real if m == 0 else (np.sqrt(2) * Y.real if cs == 'c' else np.sqrt(2) * Y.imag))
        alm = hp.map2alm(T, lmax=lmax_fit, iter=3)
        for i, (li, mi, csi) in enumerate(rb):
            a = alm[hp.Alm.getidx(lmax_fit, li, mi)]
            O[i, j] = (a.real if mi == 0 else (np.sqrt(2) * a.real if csi == 'c' else -np.sqrt(2) * a.imag))
    err = float(np.abs(O.T @ O - np.eye(len(rb))).max())
    return O, err, err < tol


def s_pm_point(C_complex, Bp, Bm, Os, tol_neg_rel=1e-12, tol_psd=1e-10):
    """Per grid point hard gates (rules v0.3 + final review C): reality, real symmetry,
    finiteness, PSD eigenvalues; then psi-set s+-; HARD FAIL on any violation."""
    wr, okr = check_reality(C_complex)
    Cr, info = to_real(C_complex)
    if not (okr and info['ok']):
        raise RuntimeError(f'HARD FAIL: reality/real-symmetry violated (worst={wr:.2e}, {info})')
    if not np.isfinite(Cr).all():
        raise RuntimeError('HARD FAIL: non-finite covariance at grid point')
    _ev = np.linalg.eigvalsh(Cr)
    if _ev.min() < -tol_psd * max(_ev.max(), 1e-300):
        raise RuntimeError(f'HARD FAIL: non-PSD covariance (lmin/lmax={_ev.min()/_ev.max():.2e})')
    sc = max(float(np.trace(Bp) + np.trace(Bm)) * float(np.trace(Cr)) / Cr.shape[0], 1e-300)
    sps, sms = [], []
    for O in Os:
        Crot = O @ Cr @ O.T
        sp = float(np.sum(Bp * Crot)); sm = float(np.sum(Bm * Crot))
        if sp < -tol_neg_rel * sc or sm < -tol_neg_rel * sc:
            raise RuntimeError(f'HARD FAIL: negative s± (implementation error): sp={sp}, sm={sm}, scale={sc}')
        sps.append(max(sp, 0.0)); sms.append(max(sm, 0.0))
    sps = np.array(sps); sms = np.array(sms)
    rpsi = (sps - sms) / np.maximum(sps + sms, 1e-300)
    return dict(splus=float(sps.mean()), sminus=float(sms.mean()),
                splus_min=float(sps.min()), splus_max=float(sps.max()),
                sminus_min=float(sms.min()), sminus_max=float(sms.max()),
                rho_psi_mean=float(rpsi.mean()), rho_psi_min=float(rpsi.min()),
                rho_psi_max=float(rpsi.max()), frac_rho_psi_neg=float(np.mean(rpsi < 0)),
                reality_worst=wr, real_ok=bool(okr and info['ok']))


def classify_rho(rho, sig_sys, rho_num=RHO_NUM):
    thr = max(3.0 * sig_sys, rho_num)
    if rho < -thr:
        return 'neg'
    if rho > thr:
        return 'pos'
    return 'unc'


def g2max_of(sminus_inf, dSmax, small_rel_scale):
    if sminus_inf <= 0:
        return np.inf, 'not_constrained_by_Sminus_budget'
    flag = 'numerical_small_sminus' if sminus_inf < SMALL_SM_REL * small_rel_scale else ''
    return dSmax / sminus_inf, flag


def direct_complex_check(C, Cr, Bp, Bm, rtol=1e-10):
    """Rules v0.3 Sec.3.1 test 4 (literal form): complex-basis quadratic expectation
    tr((M^H B M) C) must equal tr(B C_real) for both B+ and B-."""
    M, _, _ = M_matrix()
    out = {}
    for name, B in [('Bp', Bp), ('Bm', Bm)]:
        tc = float(np.real(np.trace((M.conj().T @ B @ M) @ C)))
        tr_ = float(np.trace(B @ Cr))
        out[name] = abs(tc - tr_) / max(abs(tr_), 1e-300)
    return out, all(v < rtol for v in out.values())


In [ ]:
%%writefile t2b2_run.py
"""t2b2_run v1.2 (2026-08-26c): committed analysis logic for the T2b-2 production grid.

Purpose (final pre-production review sec.4): all analysis-relevant logic (grid definition,
runner, extrapolation, refine, R2' adjacency, completion check, Git preregistration gate)
lives in this committed module, so HEAD-byte verification of {t2b2_core.py, t2b2_bridge.py,
t2b2_run.py, rules v0.3} covers the analysis; the notebook is a thin driver.

Git gate (sec.2-3): HEAD is the source of truth. Tracked-file discovery uses `git ls-files`
(never filesystem glob), content comparison uses `git show HEAD:<path>` bytes. The gate is
read-only (no pull). Untracked files therefore cannot produce a false PASS.

R2' adjacency (sec.9): S5aniso is a set of discrete anisotropic cells, not an ordered sweep;
it is excluded from adjacency and treated as isolated-candidate family (implementation freeze).

v1.1 additions (NotebookIdentity final review): head_gate records origin URL and verifies the
HEAD commit is pushed to the canonical public remote (read-only `git ls-remote`); frozen
candidate-level V7 cross-check procedure `v7_crosscheck` (existing V7 real-space validation
applied verbatim to a candidate geometry/sector; method fixed before any candidate is seen).

v1.2 (ProductionGO review sec.11, optional hardening): run_jobs skips already-done keys
per transfer tag (no duplicate rows on partial-interruption resume); completion_check counts
duplicate keys and requires zero.
"""
import os, json, csv, time, hashlib, subprocess
import numpy as np
import pandas as pd

DSMAX = 183.4   # LCDM frozen-null based conservative mean-budget diagnostic (rules v0.3 sec.3.5)
R2_EXCLUDE_FAMILIES = ('S5aniso',)


# ---------------- grid ----------------
def build_grid(subset='full'):
    SECTORS = [(1,-1,1),(1,1,-1),(1,-1,-1),(-1,1,1),(-1,-1,1),(-1,1,-1),(-1,-1,-1)]
    KCUTS = [18.0, 22.0, 26.0, 30.0, 34.0]
    BAND = (2, 3, 4)
    GEOMS = []
    def _add(family, xval, LAx, L1y, L2z, LAy):
        GEOMS.append(dict(name=f'{family}_x{xval:g}', family=family, xval=float(xval),
                          LAx=float(LAx), L1y=float(L1y), L2z=float(L2z), LAy=float(LAy)))
    for sc in [0.75, 1.0, 1.25, 1.5]:
        _add('S1scale', sc, 0.6*sc, 1.2*sc, 1.2*sc, 0.0)
    for v in [0.3, 0.45, 0.6, 0.75, 0.9]:
        _add('S2LAx', v, v, 1.2, 1.2, 0.0)
    for v in [0.6, 0.8, 1.0, 1.2, 1.4]:
        _add('S3L1y', v, 0.6, v, 1.2, 0.0)
    for f in [0.125, 0.25, 0.375, 0.5]:
        _add('S4G1f', f, 0.6, 1.2, 1.2, f*1.2)
        _add('S4G2f', f, 0.6, 0.7, 1.2, f*0.7)
    for i, (a, b, c) in enumerate([(0.6,1.2,0.6), (0.6,0.6,1.2), (0.9,0.7,1.2)]):
        _add('S5aniso', i, a, b, c, 0.0)
    if subset == 'smoke':
        GEOMS = GEOMS[:2]; SECTORS = SECTORS[:2]; KCUTS = [10.0, 12.0, 14.0, 16.0]
    else:
        fam = {}
        for g in GEOMS:
            fam[g['family']] = fam.get(g['family'], 0) + 1
        assert len(GEOMS) == 25 and fam == dict(S1scale=4, S2LAx=5, S3L1y=5,
                                                S4G1f=4, S4G2f=4, S5aniso=3), \
            f'正式grid=25幾何と不一致: {fam}'
    return GEOMS, SECTORS, KCUTS, BAND


def grid_sha(GEOMS, SECTORS, KCUTS, BAND):
    return hashlib.sha256(json.dumps(dict(GEOMS=GEOMS, SECTORS=SECTORS, KCUTS=KCUTS,
                                          BAND=list(BAND)), sort_keys=True).encode()).hexdigest()


# ---------------- Git preregistration gate (HEAD-based, read-only) ----------------
def source_only_sha(nb_bytes):
    nb = json.loads(nb_bytes.decode('utf-8'))
    canon = [dict(cell_type=c['cell_type'], source=''.join(c['source']))
             for c in nb.get('cells', [])]
    return hashlib.sha256(json.dumps(canon, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


def _git(repo, *args):
    return subprocess.run(['git', '-C', repo] + list(args), capture_output=True)


def head_gate(repo_dir, local_files, rules_basename, rules_sha_expected,
              notebook_basename=None, notebook_local_path=None, canonical_url=None):
    """local_files: {basename: local_path} to verify against HEAD bytes.
    Returns dict; ok=True only if tracked-clean AND every file is tracked in HEAD with
    byte-identical content AND the rules file in HEAD has the exact expected SHA256
    (AND, when notebook_local_path is given, its source-only SHA equals HEAD's)."""
    G = dict(ok=False, commit=None, tracked_clean=None, matches={}, rules_ok=False,
             rules_relpath=None, nb_relpath=None, nb_src_sha_head=None, nb_local_match=None,
             origin_url=None, origin_ok=None, pushed=None)
    r = _git(repo_dir, 'rev-parse', 'HEAD')
    if r.returncode != 0:
        G['error'] = 'not a git repo'
        return G
    G['commit'] = r.stdout.decode().strip()
    ru = _git(repo_dir, 'remote', 'get-url', 'origin')
    if ru.returncode == 0:
        G['origin_url'] = ru.stdout.decode().strip()
    if canonical_url is not None:
        def _norm(u):
            return (u or '').strip().rstrip('/').removesuffix('.git')
        G['origin_ok'] = bool(G['origin_url'] and _norm(G['origin_url']) == _norm(canonical_url))
        lr = _git(repo_dir, 'ls-remote', 'origin')     # read-only network query
        if lr.returncode == 0:
            remote_shas = {ln.split('\t')[0] for ln in lr.stdout.decode().splitlines() if ln}
            G['pushed'] = bool(G['commit'] in remote_shas)
        else:
            G['pushed'] = None
    G['tracked_clean'] = (_git(repo_dir, 'status', '--porcelain', '--untracked-files=no')
                          .stdout.decode().strip() == '')
    tracked = _git(repo_dir, 'ls-files').stdout.decode().splitlines()
    def _rel(basename):
        hits = [p for p in tracked if os.path.basename(p) == basename]
        return hits[0] if len(hits) == 1 else None
    def _head_bytes(relpath):
        rr = _git(repo_dir, 'show', f'HEAD:{relpath}')
        return rr.stdout if rr.returncode == 0 else None
    for base, lpath in local_files.items():
        rel = _rel(base)
        hb = _head_bytes(rel) if rel else None
        G['matches'][base] = bool(hb is not None and os.path.exists(lpath)
                                  and hashlib.sha256(hb).hexdigest()
                                  == hashlib.sha256(open(lpath, 'rb').read()).hexdigest())
    rel = _rel(rules_basename)
    G['rules_relpath'] = rel
    if rel:
        hb = _head_bytes(rel)
        G['rules_ok'] = bool(hb is not None
                             and hashlib.sha256(hb).hexdigest() == rules_sha_expected)
    nb_ok = True
    if notebook_basename:
        rel = _rel(notebook_basename)
        G['nb_relpath'] = rel
        if rel:
            hb = _head_bytes(rel)
            if hb is not None:
                try:
                    G['nb_src_sha_head'] = source_only_sha(hb)
                except Exception:
                    G['nb_src_sha_head'] = None
        nb_ok = G['nb_src_sha_head'] is not None
        if notebook_local_path and os.path.exists(notebook_local_path) and G['nb_src_sha_head']:
            G['nb_local_match'] = (source_only_sha(open(notebook_local_path, 'rb').read())
                                   == G['nb_src_sha_head'])
            nb_ok = nb_ok and bool(G['nb_local_match'])
    G['ok'] = bool(G['tracked_clean'] and all(G['matches'].values()) and G['rules_ok'] and nb_ok)
    return G


def gate_selftest(workdir, files):
    """Positive/negative self-test of head_gate (review sec.2.1/sec.7 mechanism check).
    (a) untracked-only repo must FAIL (regression for the false-pass path);
    (b) properly committed repo must PASS."""
    import shutil
    res = {}
    for mode in ['untracked', 'committed']:
        d = os.path.join(workdir, f'_gate_selftest_{mode}')
        shutil.rmtree(d, ignore_errors=True); os.makedirs(d)
        subprocess.run(['git', 'init', '-q', d], capture_output=True)
        open(os.path.join(d, 'README.md'), 'w').write('selftest')
        subprocess.run(['git', '-C', d, 'add', 'README.md'], capture_output=True)
        for base, lpath in files.items():
            open(os.path.join(d, base), 'wb').write(open(lpath, 'rb').read())
        rules_b = 'T2b2_decision_rules_frozen_v0.3.md'
        rules_sha = None
        if mode == 'committed':
            subprocess.run(['git', '-C', d, 'add', '-A'], capture_output=True)
        subprocess.run(['git', '-C', d, '-c', 'user.email=t@t', '-c', 'user.name=t',
                        'commit', '-q', '-m', 'x'], capture_output=True)
        if rules_b in files:
            rules_sha = hashlib.sha256(open(files[rules_b], 'rb').read()).hexdigest()
        g = head_gate(d, {k: v for k, v in files.items() if k != rules_b},
                      rules_b, rules_sha or '0' * 64)
        res[mode] = g['ok']
        shutil.rmtree(d, ignore_errors=True)
    return res, (res.get('untracked') is False and res.get('committed') is True)


# ---------------- runner ----------------
RAW_COLS = ['key','name','family','xval','LAx','L1y','L2z','LAy','sA','s1','s2',
            'kcut','transfer','N','A2','A3','A4','Aband','Tr',
            'splus','sminus','splus_min','splus_max','sminus_min','sminus_max',
            'rho_psi_mean','rho_psi_min','rho_psi_max','frac_rho_psi_neg','sec']


def run_jobs(jobs, ctx):
    E7, F_SW, F_full, band_stats = ctx['E7Twisted'], ctx['F_SW'], ctx['F_full'], ctx['band_stats']
    br, BP, BM, OS_LIST = ctx['br'], ctx['BP'], ctx['BM'], ctx['OS_LIST']
    CKPT, BAND = ctx['CKPT'], ctx['BAND']
    done = set()
    if os.path.exists(CKPT):
        done = set(pd.read_csv(CKPT)['key'].astype(str))
        print(f'チェックポイント再開: 既存 {len(done)} 行')
    newfile = not os.path.exists(CKPT)
    fh = open(CKPT, 'a', newline=''); w = csv.writer(fh)
    if newfile:
        w.writerow(RAW_COLS)
    t0 = time.time(); ndone = 0
    for g, s, kc in jobs:
        keys = {tag: f"{g['name']}|{s[0]},{s[1]},{s[2]}|{kc:g}|{tag}" for tag in ('SW', 'full')}
        if all(k in done for k in keys.values()):
            continue
        t1 = time.time()
        M = E7(LAx=g['LAx'], L1y=g['L1y'], L2z=g['L2z'], LAy=g['LAy'],
               sA=s[0], s1=s[1], s2=s[2], kcut=kc)
        Cs, lm = M.sky_cov_q_multi([F_SW, F_full], list(BAND), chunk=ctx.get('chunk', 3000))
        dt = time.time() - t1
        for tag, Cq in zip(('SW', 'full'), Cs):
            if keys[tag] in done:      # v1.2: 部分中断からのresumeでもduplicate行を作らない
                continue
            A, Ab, Tr, rho = band_stats(M, Cq, lm, BAND)
            sr = br.s_pm_point(Cq, BP, BM, OS_LIST)   # per-point hard gates内蔵
            w.writerow([keys[tag], g['name'], g['family'], g['xval'],
                        g['LAx'], g['L1y'], g['L2z'], g['LAy'], s[0], s[1], s[2],
                        kc, tag, M.N, A[2], A[3], A[4], Ab, Tr,
                        sr['splus'], sr['sminus'], sr['splus_min'], sr['splus_max'],
                        sr['sminus_min'], sr['sminus_max'], sr['rho_psi_mean'],
                        sr['rho_psi_min'], sr['rho_psi_max'], sr['frac_rho_psi_neg'],
                        round(dt, 2)])
            done.add(keys[tag])
        fh.flush(); ndone += 1
        if ndone == 3:
            per = (time.time() - t0) / 3
            print(f'  ETA目安: {per:.1f}s/ladder点 × 残り{len(jobs)-3} ≈ {(len(jobs)-3)*per/60:.0f}分')
    fh.close()


# ---------------- extrapolation / classification (rules v0.3 sec.3.3) ----------------
def _fit_p(kc, v, p):
    x = np.asarray(kc, float) ** (-p)
    Am = np.stack([np.ones_like(x), x], 1)
    cf, *_ = np.linalg.lstsq(Am, np.asarray(v, float), rcond=None)
    return float(cf[0]), float(np.sqrt(np.mean((Am @ cf - v) ** 2)))


def run_extrapolation(CKPT, br):
    df = pd.read_csv(CKPT).drop_duplicates(subset=['key'], keep='last')
    recs = []
    for keytuple, gdf in df.groupby(['name','family','xval','sA','s1','s2','transfer']):
        name, fam, xv, sA, s1, s2, tag = keytuple
        gdf = gdf.sort_values('kcut')
        if len(gdf) < 4:
            continue
        kc = gdf['kcut'].values
        rec = dict(name=name, family=fam, xval=xv, sA=sA, s1=s1, s2=s2,
                   transfer=tag, kmax=float(kc.max()), npts=len(gdf))
        for col in ['A2','A3','A4','Aband','Tr']:   # legacy（v0.1定義：RMS込み・副次）
            v = gdf[col].values
            E1, r1 = _fit_p(kc, v, 3.0); E2, _ = _fit_p(kc, v, 2.0); E3 = float(v[-1])
            sysd = max(abs(E1-E2), abs(E1-E3), r1)
            rec[col+'_inf'] = E1; rec[col+'_sys'] = sysd
            rec[col+'_class'] = ('pos' if (E1-3*sysd > 0 and E2 > 0 and E3 > 0)
                                 else ('neg' if (E1+3*sysd < 0 and E2 < 0 and E3 < 0) else 'unc'))
        mods = {}
        for col in ['splus','sminus']:
            v = gdf[col].values
            E1, r1 = _fit_p(kc, v, 3.0); E2, _ = _fit_p(kc, v, 2.0); E3 = float(v[-1])
            mods[col] = dict(E1=E1, E2=E2, E3=E3, rms=r1)
            rec[col+'_inf'] = E1
            rec[col+'_sys'] = max(abs(E1-E2), abs(E1-E3))   # 凍結v0.3と完全一致
            rec[col+'_fitrms'] = r1
        rho_e = {}
        for e in ['E1','E2','E3']:
            den = mods['splus'][e] + mods['sminus'][e]
            rho_e[e] = (mods['splus'][e] - mods['sminus'][e]) / den if den > 0 else np.nan
        rec['rho_q_inf'] = rho_e['E1']
        rec['rho_q_sys'] = float(np.nanmax([abs(rho_e['E2']-rho_e['E1']),
                                            abs(rho_e['E3']-rho_e['E1'])]))
        rec['s_tot_inf'] = mods['splus']['E1'] + mods['sminus']['E1']
        eps_x = 1e-6 * max(mods['splus']['E3'] + mods['sminus']['E3'], 1e-300)
        unstable = False
        for e in ['E1','E2','E3']:
            den_e = mods['splus'][e] + mods['sminus'][e]
            unstable |= (mods['splus'][e] < -eps_x or mods['sminus'][e] < -eps_x or den_e <= 0
                         or (not np.isfinite(rho_e[e])) or abs(rho_e[e]) > 1 + 1e-9)
        rec['extrap_status'] = 'unstable' if unstable else 'ok'
        rec['rho_inf_legacy'] = 4*np.pi*rec['Aband_inf']/rec['Tr_inf']
        recs.append(rec)
    ext = pd.DataFrame(recs)
    if len(ext) == 0:
        return ext
    smax = {t: ext[ext.transfer == t]['s_tot_inf'].max() for t in ext.transfer.unique()}
    def _cls(r):
        if r['extrap_status'] == 'unstable':
            return 'unc'
        if not np.isfinite(r['rho_q_inf']) or r['s_tot_inf'] <= 0 \
           or r['s_tot_inf'] < br.NOSIG_REL * smax[r['transfer']]:
            return 'no-signal'
        return br.classify_rho(r['rho_q_inf'], r['rho_q_sys'])
    ext['rho_q_class'] = ext.apply(_cls, axis=1)
    ext.loc[ext.rho_q_class == 'no-signal', 'rho_q_inf'] = np.nan
    med_sm = {t: ext[ext.transfer == t]['sminus_inf'].clip(lower=0).median()
              for t in ext.transfer.unique()}
    g2s, mv_p, mv_m, flags = [], [], [], []
    for _, r in ext.iterrows():
        smv = max(r['sminus_inf'], 0.0)
        if r['sminus_inf'] <= 0:                       # minor fix（レビュー§11）
            g2s.append(np.inf); flags.append('not_constrained_by_Sminus_budget')
            mv_p.append(np.nan); mv_m.append(0.0)      # s-=0なら任意有限g_effでS-移動は0
        else:
            g2, fl = br.g2max_of(smv, DSMAX, med_sm[r['transfer']])
            g2s.append(g2); flags.append(fl)
            mv_p.append(g2 * max(r['splus_inf'], 0.0)); mv_m.append(g2 * smv)
    ext['g2eff_max_meanbudget'] = g2s; ext['g2eff_flag'] = flags
    ext['move_splus_at_g2eff'] = mv_p; ext['move_sminus_at_g2eff'] = mv_m
    return ext


def find_R2_pairs(ext, GEOMS, exclude=R2_EXCLUDE_FAMILIES):
    """final tableに対する隣接negペア判定（S5anisoは除外・isolated-candidate family扱い）"""
    fu = ext[(ext.transfer == 'full') & (ext.rho_q_class == 'neg')
             & ~ext.family.isin(exclude)]
    xv = {}
    for g in GEOMS:
        xv.setdefault(g['family'], []).append(g['xval'])
    pairs = []
    for (fam, sa, s1_, s2_), gsub in fu.groupby(['family','sA','s1','s2']):
        xs = set(gsub['xval']); seq = sorted(xv.get(fam, []))
        for a, b in zip(seq, seq[1:]):
            if a in xs and b in xs:
                pairs.append((fam, int(sa), int(s1_), int(s2_), a, b))
    return pairs


def refine_pass(ext, GEOMS, ctx, run_mode):
    """主refine trigger＝ρ_q neg/unc・unstableのみ → refine ladder → 自動再外挿 →
    final分類 → **final R2隣接判定**（レビュー§8：R2′はfinal tableのみで判定）"""
    refine_expected = set()
    if run_mode == 'smoke' or len(ext) == 0:
        print('（refineスキップ：smokeまたは外挿結果なし）')
        return ext, find_R2_pairs(ext, GEOMS) if len(ext) else [], refine_expected
    flagged = ext[(ext.transfer == 'full') & ((ext.rho_q_class.isin(['neg','unc'])) |
                                              (ext.extrap_status == 'unstable'))]
    legacy = ext[(ext.transfer == 'full') & (ext.Aband_class != 'pos')
                 & ~ext.index.isin(flagged.index)]
    if len(legacy):
        print(f'（参考）旧Aband由来candidate {len(legacy)}件：secondary legacy diagnosticのため'
              '主refineには含めない')
    cfgs = flagged[['name','sA','s1','s2']].drop_duplicates()
    print(f'精査対象構成: {len(cfgs)}')
    if len(cfgs):
        gmap = {g['name']: g for g in GEOMS}
        jobs2 = [(gmap[r['name']], (int(r['sA']), int(r['s1']), int(r['s2'])), kc)
                 for _, r in cfgs.iterrows() for kc in [38.0, 42.0]]
        for _, r in cfgs.iterrows():
            for kc in [38.0, 42.0]:
                for tag in ('SW', 'full'):
                    refine_expected.add(f"{r['name']}|{r['sA']},{r['s1']},{r['s2']}|{kc:g}|{tag}")
        run_jobs(jobs2, ctx)
        ext = run_extrapolation(ctx['CKPT'], ctx['br'])
    r2 = find_R2_pairs(ext, GEOMS)
    return ext, r2, refine_expected


def completion_check(CKPT, GEOMS, SECTORS, KCUTS, refine_expected=set()):
    """R1′完了証明用：期待keyの完全一致検査（missing=0・unexpected=0）"""
    expected = {f"{g['name']}|{s[0]},{s[1]},{s[2]}|{kc:g}|{tag}"
                for g in GEOMS for s in SECTORS for kc in KCUTS for tag in ('SW', 'full')}
    expected |= set(refine_expected)
    if os.path.exists(CKPT):
        _k = pd.read_csv(CKPT)['key'].astype(str)
        actual = set(_k)
        n_dup = int(len(_k) - len(actual))
    else:
        actual = set(); n_dup = 0
    return dict(n_expected=len(expected), n_actual=len(actual), n_duplicates=n_dup,
                missing=sorted(expected - actual)[:20], n_missing=len(expected - actual),
                unexpected=sorted(actual - expected)[:20], n_unexpected=len(actual - expected),
                ok=bool(expected == actual and n_dup == 0))


def v7_crosscheck(E7cls, F_SW, geom, sector, kcut=14.0, npts=3000, tol=1e-2):
    """Frozen candidate-level V7 cross-check (procedure fixed BEFORE any candidate is seen):
    apply the existing V7 real-space validation verbatim to the candidate (geometry, sector):
    harmonic sum_l A_l (SW transfer, l=0..30) vs real-space mean(2 D^2)/25 over a Fibonacci
    sphere; PASS iff |ratio-1| < tol."""
    i = np.arange(npts) + 0.5
    z = 1 - 2 * i / npts
    r = np.sqrt(1 - z * z)
    th = np.pi * (1 + 5 ** 0.5) * i
    pts = np.stack([r * np.cos(th), r * np.sin(th), z], 1)
    M = E7cls(LAx=geom['LAx'], L1y=geom['L1y'], L2z=geom['L2z'], LAy=geom['LAy'],
              sA=sector[0], s1=sector[1], s2=sector[2], kcut=kcut)
    C, lm = M.sky_cov_q(F_SW, list(range(0, 31)), chunk=500)
    A = M.A_refl_y(C, lm, list(range(0, 31)))
    D = M.Cphi_pairs(pts, pts * np.array([1.0, -1.0, 1.0]))
    ratio = float(sum(A.values()) / (np.mean(2 * D * D) / 25.0))
    return dict(ratio=ratio, ok=bool(abs(ratio - 1) < tol), kcut=kcut, npts=npts, tol=tol)


In [ ]:
# === 事前登録ゲート v2（HEADベース・read-only・self-test付き）===
import importlib, t2b2_run
importlib.reload(t2b2_run)
import t2b2_run as tr
import subprocess
T2B2_REPO_URL = 'https://github.com/tsujikeita/mirror-topology.git'
NOTEBOOK_BASENAME = 'MirrorTopology_T2b2_signmap_v0.6.ipynb'
NOTEBOOK_PATH = globals().get('NOTEBOOK_PATH', None)   # 未指定ならrepo内のtracked notebookへ決定的に自動解決
RULES_SHA_EXPECTED = '8fa8bc2395a838e89107701131d8fb123f422a55542999942ed4d9d4b81697ae'
_CANDS = ([os.path.join(BASE, 'mirror_topology_repo'), '/content/mt_repo'] if IN_COLAB
          else ['/home/claude/mt_repo'])
T2B2_REPO_DIR = next((p for p in _CANDS if os.path.isdir(os.path.join(p, '.git'))), _CANDS[-1])
if not os.path.isdir(os.path.join(T2B2_REPO_DIR, '.git')):
    _env = dict(os.environ, GIT_TERMINAL_PROMPT='0')
    r = subprocess.run(['git', 'clone', '--depth', '5', T2B2_REPO_URL, T2B2_REPO_DIR],
                       capture_output=True, env=_env)
    print('fresh clone:', 'OK' if r.returncode == 0 else r.stderr.decode()[-200:])
# ゲートはread-only：既存checkoutに対してpullは行わない（レビュー§6）
if NOTEBOOK_PATH is None:      # ProductionGOレビュー§8：commit後のsource編集を不要にする決定的解決
    _rr = subprocess.run(['git', '-C', T2B2_REPO_DIR, 'ls-files'], capture_output=True, text=True)
    _hits = [p for p in _rr.stdout.splitlines() if os.path.basename(p) == NOTEBOOK_BASENAME]
    if len(_hits) == 1:
        NOTEBOOK_PATH = os.path.join(T2B2_REPO_DIR, _hits[0])
        print('NOTEBOOK_PATH 自動解決:', NOTEBOOK_PATH)
REPO_GATE = tr.head_gate(T2B2_REPO_DIR,
                         dict((f, f) for f in ['t2b2_core.py', 't2b2_bridge.py', 't2b2_run.py']),
                         'T2b2_decision_rules_frozen_v0.3.md', RULES_SHA_EXPECTED,
                         notebook_basename=NOTEBOOK_BASENAME, notebook_local_path=NOTEBOOK_PATH,
                         canonical_url=T2B2_REPO_URL)
print('REPO_GATE =', {k: (v[:12] + '…' if isinstance(v, str) and len(v) > 20 else v)
                      for k, v in REPO_GATE.items()})
if globals().get('SUBSET', 'full') == 'smoke':
    ST, ST_OK = tr.gate_selftest(os.getcwd(),
        {'t2b2_core.py': 't2b2_core.py', 't2b2_bridge.py': 't2b2_bridge.py',
         't2b2_run.py': 't2b2_run.py',
         'T2b2_decision_rules_frozen_v0.3.md':
             os.path.join(T2B2_REPO_DIR, 'docs', 'T2b2_decision_rules_frozen_v0.3.md')})
    print('gate self-test（untracked→FAIL / committed→PASS）:', ST, '=>', 'OK' if ST_OK else 'NG')
    assert ST_OK, 'ゲートself-test不合格（false-pass回帰）'
else:
    ST = None
print('※production起動にはREPO_GATE.ok=Trueが必須。NOTEBOOK_PATH指定でnotebook identityも照合。')

In [ ]:
# === 検証ゲート（不合格なら停止：検証済みコードのみ本番へ） ===
import importlib, numpy as np, time
import t2b2_core; importlib.reload(t2b2_core)
from t2b2_core import E7Twisted, F_SW, band_stats, extrapolate_k3

def fib_sphere(n):
    i = np.arange(n)+0.5
    z = 1-2*i/n; r = np.sqrt(1-z*z); th = np.pi*(1+5**0.5)*i
    return np.stack([r*np.cos(th), r*np.sin(th), z],1)

t0 = time.time(); rng = np.random.default_rng(1)
X = rng.uniform(-0.8,0.8,(10,3)); Y = rng.uniform(-0.8,0.8,(10,3))
gate = dict(deck=0.0, route=0.0, gram=0.0)
for geo, s in [ (dict(LAx=0.6,L1y=1.2,L2z=1.2,LAy=0.0), (1,-1,1)),
                (dict(LAx=0.6,L1y=1.2,L2z=1.2,LAy=0.0), (-1,-1,1)),
                (dict(LAx=0.6,L1y=1.2,L2z=1.2,LAy=0.3), (1,-1,1)),
                (dict(LAx=0.6,L1y=1.2,L2z=1.2,LAy=0.3), (-1,1,-1)) ]:
    M = E7Twisted(**geo, sA=s[0], s1=s[1], s2=s[2], kcut=12.0)
    gate['deck'] = max(gate['deck'], float(M.deck_check()))
    Ci = M.Cphi_image(X, Y)
    gate['route'] = max(gate['route'], float(np.abs(M.Cphi_modes(X,Y)-Ci).max()/np.abs(Ci).max()))
    G = M.gram()
    gate['gram'] = max(gate['gram'], float(max(np.abs(np.diag(G)-1).max(),
                                               np.abs(G-np.diag(np.diag(G))).max())))
# end-to-end（縮小V7）：ℓ空間パイプライン vs 実空間求積
M = E7Twisted(LAx=0.6,L1y=1.2,L2z=1.2,LAy=0.0,sA=1,s1=-1,s2=1,kcut=10.0)
C, lm = M.sky_cov_q(F_SW, list(range(0,31)), chunk=500)
A = M.A_refl_y(C, lm, list(range(0,31)))
pts = fib_sphere(3000); D = M.Cphi_pairs(pts, pts*np.array([1.0,-1.0,1.0]))
gate['V7ratio'] = float(sum(A.values())/(np.mean(2*D*D)/25.0))
# MC簡易照合（局所正規順序で C_q=2C_phi^2）
Xs = rng.uniform(-0.7,0.7,(6,3)); Cth = M.Cphi_modes(Xs, Xs)
phi = M.sample(Xs, 40000, rng); qf = phi**2 - np.diag(Cth)[:,None]
gate['MCq'] = float(np.abs(qf@qf.T/qf.shape[1] - 2*Cth**2).max()/(2*Cth**2).max())
ok = (gate['deck']<1e-10 and gate['route']<1e-10 and gate['gram']<1e-10
      and abs(gate['V7ratio']-1)<1e-3 and gate['MCq']<0.08)
print({k:(f'{v:.2e}' if k!='V7ratio' else f'{v:.6f}') for k,v in gate.items()}, f'({time.time()-t0:.0f}s)')
if not ok:
    raise RuntimeError('検証ゲート不合格：本番実行を中止します。 ' + str(gate))
VALID_GATE = dict(gate)
print('=== GATE PASS ===')

In [ ]:
# === 橋渡し検証電池 v0.3（規則§3.1完全版＋ψ収束＋direct-complex）：不合格なら停止 ===
import importlib, numpy as np, healpy as hp, time
import t2b2_bridge; importlib.reload(t2b2_bridge)
import t2b2_bridge as br
t0 = time.time()
Zb = np.load(BPM_PATH, allow_pickle=True)
BP, BM = Zb['Bp'], Zb['Bm']
rb_npz = [(int(l), int(m), str(c)) for (l, m, c) in Zb['basis_lm']]
assert rb_npz == [(l, m, c) for (l, m, c) in br.real_basis_lm()], 'T3: B±基底順序不一致'
Mt = E7Twisted(LAx=0.6, L1y=1.2, L2z=1.2, LAy=0.0, sA=1, s1=-1, s2=1, kcut=10.0)
Ct, lmt = Mt.sky_cov_q(F_SW, [2, 3, 4], chunk=500)
assert lmt == br.lm_full(), 'lm順序不一致'
w0, ok0 = br.check_reality(Ct)
Cr, info = br.to_real(Ct)
ev = np.linalg.eigvalsh(Cr)
trc = abs(np.trace(Cr) - np.trace(Ct).real) / abs(np.trace(Ct).real)
w4, ok4 = br.twopoint_check(Ct, Cr)
dc, okdc = br.direct_complex_check(Ct, Cr, BP, BM)          # 規則§3.1テスト4（文字どおり）
Mm, _, _ = br.M_matrix(); rngb = np.random.default_rng(5); ok5 = True
for _ in range(5):
    A0 = rngb.standard_normal((21, 21)); C0 = A0 @ A0.T
    Cc = Mm.conj().T @ C0 @ Mm
    okr = br.check_reality(Cc)[1]; C0b, inf2 = br.to_real(Cc)
    ok5 &= okr and inf2['ok'] and np.abs(C0b - C0).max() / np.abs(C0).max() < 1e-12
# ψ収束（8/16/32）：16=厳密（|m-m'|<=8<16）を毎回確認
def _spm_n(n):
    d = br.frozen_axis_vec(); ref = np.array([0., 0., 1.])
    e1_0 = np.cross(ref, d); e1_0 /= np.linalg.norm(e1_0); e3_0 = np.cross(e1_0, d)
    sps, sms, frames = [], [], []
    for k in range(n):
        psi = 2 * np.pi * k / n
        e1 = np.cos(psi) * e1_0 + np.sin(psi) * e3_0
        R3 = np.stack([e1, d, np.cross(e1, d)], 1)
        O, e, okO = br.rotation_O(R3); assert okO
        frames.append(O)
        Crot = O @ Cr @ O.T
        sps.append(np.sum(BP * Crot)); sms.append(np.sum(BM * Crot))
    return float(np.mean(sps)), float(np.mean(sms)), frames
s8 = _spm_n(8); s16 = _spm_n(16); s32 = _spm_n(32)
OS_LIST = s16[2]                                             # 本番: N_PSI=16（厳密）
psi_conv = dict(rel_16_32=max(abs(s16[0]-s32[0])/s32[0], abs(s16[1]-s32[1])/s32[1]),
                alias_8=max(abs(s8[0]-s16[0])/s16[0], abs(s8[1]-s16[1])/s16[1]))
even_idx = [i for i, (l, m, cs) in enumerate(br.real_basis_lm()) if cs == 'c']
xe = np.zeros(21); xe[even_idx] = rngb.standard_normal(len(even_idx))
xr = OS_LIST[3] @ xe; dax = br.frozen_axis_vec()
Vv = rngb.standard_normal((400, 3)); Vv /= np.linalg.norm(Vv, axis=1)[:, None]
RVv = Vv - 2 * (Vv @ dax)[:, None] * dax[None, :]
def _evalT(x, V):
    th, ph = hp.vec2ang(V); T = np.zeros(len(th))
    for i, (l, m, cs) in enumerate(br.real_basis_lm()):
        Y = br._Ylm(l, m, th, ph)
        T += x[i] * (Y.real if m == 0 else (np.sqrt(2) * Y.real if cs == 'c' else np.sqrt(2) * Y.imag))
    return T
dmax = np.abs(_evalT(xr, Vv) - _evalT(xr, RVv)).max() / np.abs(_evalT(xr, Vv)).max()
res0 = br.s_pm_point(Ct, BP, BM, OS_LIST)
L = np.linalg.cholesky(Cr + 1e-12 * np.eye(21) * np.abs(Cr).max())
xs = OS_LIST[0] @ (L @ rngb.standard_normal((21, 4000)))
mc_ratio = float(np.einsum('ip,ij,jp->p', xs, BP, xs, optimize=True).mean()
                 / np.sum(BP * (OS_LIST[0] @ Cr @ OS_LIST[0].T)))
tests = dict(T0_reality=w0, T1_imag=info['imag_rel'], T1_asym=info['asym_rel'],
             T2_psd_ratio=float(ev.min() / ev.max()), T4_twopoint=w4, T4_trace=trc,
             T4_direct_Bp=dc['Bp'], T4_direct_Bm=dc['Bm'], T5_roundtrip=bool(ok5),
             PSI_rel_16_32=psi_conv['rel_16_32'], PSI_alias_8=psi_conv['alias_8'],
             T7_alignnull=dmax, T6_mc_ratio=mc_ratio)
print({k: (f'{v:.2e}' if isinstance(v, float) else v) for k, v in tests.items()})
OKB = (ok0 and info['ok'] and ev.min() > -1e-10 * ev.max() and ok4 and trc < 1e-12 and okdc
       and ok5 and psi_conv['rel_16_32'] < 1e-10 and dmax < 3e-5 and abs(mc_ratio - 1) < 0.05
       and res0['real_ok'] and br.N_PSI == 16)
print(f"s±サニティ(kcut=10,SW): s+={res0['splus']:.3e} s-={res0['sminus']:.3e} "
      f"ρψ[{res0['rho_psi_min']:+.3f},{res0['rho_psi_max']:+.3f}] ({time.time()-t0:.0f}s)")
if not OKB:
    raise RuntimeError('橋渡し電池 不合格：本番実行を中止します。 ' + str(tests))
VALID_BRIDGE = {k: (float(v) if isinstance(v, (int, float)) else v) for k, v in tests.items()}
print('=== BRIDGE BATTERY v0.5 PASS ===')

In [ ]:
# === CAMB full transfer v0.3（provenance-lockedキャッシュ＋検証：C_ℓ再構成・SW極限） ===
import numpy as np, os, hashlib, json
FIDUCIAL = dict(H0=67.36, ombh2=0.02237, omch2=0.1200, tau=0.0544, As=2.1e-9, ns=0.9649)
import camb
CAMB_CACHE = os.path.join(OUT, 'camb_transfer_cache_v2.npz')
def _build_cache():
    pars = camb.set_params(lmax=20, lens_potential_accuracy=0, **FIDUCIAL)
    res = camb.get_transfer_functions(pars)
    td = res.get_cmb_transfer_data('scalar')
    L_list = np.array(getattr(td, 'L', getattr(td, 'l', None)), int)
    q_grid = np.array(td.q)
    delta0 = np.array(td.delta_p_l_k)[0]
    chistar = float(res.comoving_radial_distance(res.get_derived_params()['zstar']))
    meta = json.dumps(dict(fiducial=FIDUCIAL, camb_version=camb.__version__, lmax=20,
                           lens_potential_accuracy=0, transfer='scalar', component_index=0),
                      sort_keys=True)
    np.savez(CAMB_CACHE, q=q_grid, L=L_list, delta0=delta0, chistar=chistar, meta=meta)
    return q_grid, L_list, delta0, chistar
need = True
if os.path.exists(CAMB_CACHE):
    z = np.load(CAMB_CACHE, allow_pickle=True)
    meta_now = json.dumps(dict(fiducial=FIDUCIAL, camb_version=camb.__version__, lmax=20,
                               lens_potential_accuracy=0, transfer='scalar', component_index=0),
                          sort_keys=True)
    if 'meta' in z.files and str(z['meta']) == meta_now:
        q_grid, L_list, delta0, chistar = z['q'], z['L'].astype(int), z['delta0'], float(z['chistar'])
        need = False; print('CAMBキャッシュ読込（provenance一致）:', CAMB_CACHE)
    else:
        print('CAMBキャッシュのprovenance不一致 → 再計算')
if need:
    q_grid, L_list, delta0, chistar = _build_cache(); print('CAMB計算・キャッシュ保存:', CAMB_CACHE)
CAMB_CACHE_SHA = hashlib.sha256(open(CAMB_CACHE, 'rb').read()).hexdigest()
iL = {int(l): i for i, l in enumerate(L_list)}
def F_full(l, K):
    return np.interp(np.asarray(K) / chistar, q_grid, delta0[iL[int(l)]])
# --- 検証1: C_ℓ再構成 vs CAMB raw C_ℓ（unlensed scalar・無次元） ---
pars2 = camb.set_params(lmax=30, lens_potential_accuracy=0, **FIDUCIAL)
res2 = camb.get_results(pars2)
clraw = res2.get_cmb_power_spectra(pars2, raw_cl=True)['unlensed_scalar'][:, 0]
k0 = 0.05
PR = FIDUCIAL['As'] * (q_grid / k0) ** (FIDUCIAL['ns'] - 1)
CAMB_ARR_SHA = hashlib.sha256(q_grid.tobytes() + delta0.tobytes()).hexdigest()
CAMB_VALID = dict(cl_ratio={}, f_ratio={})
okC = True
for l in [2, 3, 4]:
    D = delta0[iL[l]]
    Cl_rec = 4 * np.pi * np.trapezoid(PR * D * D / q_grid, q_grid)
    r = Cl_rec / clraw[l]
    okC &= abs(r - 1) < 0.03
    CAMB_VALID['cl_ratio'][l] = float(r)
    print(f'  C_{l} 再構成/CAMB = {r:.4f}')
assert okC, 'full transfer検証失敗（C_ℓ再構成の不一致）'
# --- 検証2: SW極限の符号一貫性 ---
sgn = []
for l in [2, 3, 4]:
    r = F_full(l, 1.0) / F_SW(l, 1.0)
    sgn.append(np.sign(r)); CAMB_VALID['f_ratio'][l] = float(r)
    print(f'  F_full/F_SW (l={l}, K=1) = {r:+.3f}')
assert len(set(sgn)) == 1, 'SW極限で相対符号が不一致'
print(f'chistar = {chistar:.1f} Mpc / CAMB {camb.__version__} / 検証済み（TODO-verify閉鎖）')

In [ ]:
# === グリッド定義（凍結：t2b2_run.build_grid） ===
SUBSET = globals().get('SUBSET', 'full')
GEOMS, SECTORS, KCUTS, BAND = tr.build_grid(SUBSET)
GRID_SHA = tr.grid_sha(GEOMS, SECTORS, KCUTS, BAND)
print(f'geometries={len(GEOMS)}  sectors={len(SECTORS)}  kcuts={len(KCUTS)}'
      f'  -> ladder-configs={len(GEOMS)*len(SECTORS)}')
print('GRID_SHA =', GRID_SHA[:12], '…')

In [ ]:
# === RUN_DIR（config-hash束縛・commit-first enforcement） ===
import hashlib, json
import numpy as _np, scipy as _sp, healpy as _hpv, camb as _cbv, sys as _sys
RUN_MODE = 'smoke' if SUBSET == 'smoke' else 'production'
if RUN_MODE == 'production':
    assert NOTEBOOK_PATH is not None and os.path.exists(NOTEBOOK_PATH), \
        'productionではNOTEBOOK_PATHが必須です（Drive clone内の本ノートブックのパスを指定）'
    assert REPO_GATE['ok'], ('production前ゲート不合格：規則v0.3・notebook v0.6・core・bridge・'
                             'runをmirror-topologyへcommitし（tracked clean・HEAD一致），再実行してください')
    assert REPO_GATE['nb_local_match'] is True, \
        'notebook identity不一致：実行中ノートブックがHEADのnotebookと異なります'
    assert REPO_GATE.get('origin_ok') is True and REPO_GATE.get('pushed') is True, \
        f"public remote検証不合格（origin_ok={REPO_GATE.get('origin_ok')}, pushed={REPO_GATE.get('pushed')}）：正準URLへpushしてください" 
CONFIG = dict(core_sha=hashlib.sha256(open('t2b2_core.py','rb').read()).hexdigest(),
              bridge_sha=hashlib.sha256(open('t2b2_bridge.py','rb').read()).hexdigest(),
              run_sha=hashlib.sha256(open('t2b2_run.py','rb').read()).hexdigest(),
              bpm_array_sha=BPM_ARRAY_SHA256, camb_cache_sha=CAMB_CACHE_SHA,
              grid_sha=GRID_SHA, N_PSI=br.N_PSI, RHO_NUM=br.RHO_NUM, NOSIG_REL=br.NOSIG_REL,
              rules='T2b2_decision_rules_frozen_v0.3', rules_sha=RULES_SHA_EXPECTED,
              t2b2_repo_commit=REPO_GATE['commit'], nb_src_sha_head=REPO_GATE['nb_src_sha_head'],
              DSMAX=tr.DSMAX,
              versions=dict(python=_sys.version.split()[0], numpy=_np.__version__,
                            scipy=_sp.__version__, healpy=_hpv.__version__, camb=_cbv.__version__),
              mode=RUN_MODE)
CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()
RUN_DIR = os.path.join(OUT, f'{RUN_MODE}_{CONFIG_HASH[:12]}')
os.makedirs(RUN_DIR, exist_ok=True)
MANIFEST = os.path.join(RUN_DIR, 'config_manifest.json')
if os.path.exists(MANIFEST):
    prev = json.load(open(MANIFEST))
    assert prev == CONFIG, 'RUN_DIRのconfig manifest不一致：再開拒否'
    print('既存RUN_DIRのconfig一致：checkpoint再開許可')
else:
    json.dump(CONFIG, open(MANIFEST, 'w'), indent=2)
CKPT = os.path.join(RUN_DIR, 't2b2_grid_raw.csv')
print('RUN_DIR =', RUN_DIR, f'({RUN_MODE})')

In [ ]:
# === ランナー（t2b2_run.run_jobs） ===
CTX = dict(E7Twisted=E7Twisted, F_SW=F_SW, F_full=F_full, band_stats=band_stats,
           br=br, BP=BP, BM=BM, OS_LIST=OS_LIST, CKPT=CKPT, BAND=BAND, chunk=3000)
jobs = [(g, s, kc) for g in GEOMS for s in SECTORS for kc in KCUTS]
tr.run_jobs(jobs, CTX)
print('runner done:', CKPT)

In [ ]:
# === 初期外挿（final判定はrefine後に実施） ===
ext = tr.run_extrapolation(CKPT, br)
if len(ext) == 0:
    print('（外挿対象なし）')
else:
    fu = ext[ext.transfer == 'full']
    print('=== full transfer: ρ_q（凍結統計）初期分類 ===', fu['rho_q_class'].value_counts().to_dict())
    print('（3×σ_sysは数値安定性基準・p系はfrozen-axis conditional・smokeの分類は科学結果ではない）')
    r2_init = tr.find_R2_pairs(ext, GEOMS)
    print('R2′隣接negペア（初期・非final・参考値）:', r2_init if r2_init else 'なし')

In [ ]:
# === refine → 自動再外挿 → final分類 → final R2判定 → 完了検査 ===
ext, R2_FINAL, refine_expected = tr.refine_pass(ext, GEOMS, CTX, RUN_MODE)
EXT = os.path.join(RUN_DIR, 't2b2_extrapolated.csv')
if len(ext):
    ext.to_csv(EXT, index=False)
    fu = ext[ext.transfer == 'full']
    print('=== final分類 ===', fu['rho_q_class'].value_counts().to_dict())
    print('R2′隣接negペア（final・S5aniso除外）:', R2_FINAL if R2_FINAL else 'なし',
          '（成立にはさらにV7実空間cross-checkが必要）')
    cand = fu[fu['rho_q_class'] == 'neg']
    if len(cand):
        print(cand[['name','sA','s1','s2','rho_q_inf','rho_q_sys','g2eff_max_meanbudget',
                    'move_sminus_at_g2eff']].to_string())
COMPLETION = tr.completion_check(CKPT, GEOMS, SECTORS, KCUTS, refine_expected)
print('完了検査（R1′）:', {k: COMPLETION[k] for k in ['n_expected','n_actual','n_missing','n_unexpected','ok']})
if RUN_MODE == 'production':
    assert COMPLETION['ok'], f'完了検査不合格: {COMPLETION}'
print('saved(final):', EXT)

In [ ]:
# === 図：凍結統計 ρ_q∞ と s±∞（掃引族ごと・full transfer） ===
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
if len(ext):
    fu = ext[(ext.transfer=='full') & (ext.rho_q_class != 'no-signal')]
    fams = sorted(fu.family.unique())
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for fam in fams:
        sub = fu[fu.family == fam].sort_values('xval')
        for sec, ss in sub.groupby(['sA','s1','s2']):
            axes[0].plot(ss.xval, ss.rho_q_inf, 'o-', ms=3, alpha=0.6)
            axes[1].plot(ss.xval, ss.splus_inf/ss.sminus_inf.clip(lower=1e-30), 'o-', ms=3, alpha=0.6)
    axes[0].axhline(0, color='k', lw=0.5); axes[0].set_ylabel('rho_q_inf (frozen statistic)')
    axes[1].set_yscale('log'); axes[1].set_ylabel('s+_inf / s-_inf')
    for ax in axes: ax.set_xlabel('xval')
    fig.suptitle('T2b-2 v0.6: frozen-statistic secondary observables (full transfer)')
    fig.tight_layout(); fig.savefig(os.path.join(RUN_DIR, 't2b2_rhoq_map.png'), dpi=110)
    print('figure saved:', os.path.join(RUN_DIR, 't2b2_rhoq_map.png'))


## 判定規則の参照と実装凍結（v0.6）
準拠：**T2b2_decision_rules_frozen_v0.3**（HEADバイトSHA=8fa8bc23…を厳密照合）。
B±同一性＝科学配列hash 9693b207…（source of truth）。ψ周辺化＝N_PSI=16（ℓ≤4で厳密）・
主判定は事前凍結の一様ψ平均。解析ロジックは全てコミット対象（core v0.3・bridge v1.2・
**run v1.2**）でHEADバイト照合の対象。**R2′判定はrefine後のfinal tableのみ**で行い，
**S5anisoは隣接判定から除外**（順序掃引でないため・isolated-candidate family——実装凍結）。
R2′成立には隣接2点neg＋refine生存＋V7実空間cross-checkの3条件。完了検査（期待key完全一致）が
R1′完了証明に対応。g_eff²は規格化を全吸収した有効振幅。smokeの分類は科学結果ではない。

**v0.6追加凍結**：production＝NOTEBOOK_PATH必須・nb_local_match必須・origin正準URL＋
push済み必須（証拠鎖：public remote commit→clone→exact HEAD verification→production）。
R2′のV7 cross-check手順は`t2b2_run.v7_crosscheck`（SW・ℓ≤30・|ratio−1|<1e-2・候補の
(幾何,セクター)へ既存検証をそのまま適用）として**候補を見る前に**コミット凍結。

In [ ]:
# === 最終provenance v0.6 ===
import hashlib, json, datetime, sys
def _sha(p):
    return hashlib.sha256(open(p, 'rb').read()).hexdigest() if os.path.exists(p) else None
import scipy, camb as _cb
prov = dict(notebook='T2b2 signmap v0.6', date=str(datetime.date.today()), run_mode=RUN_MODE,
            rules='T2b2_decision_rules_frozen_v0.3 (approved 2026-08-26)',
            config=CONFIG, config_hash=CONFIG_HASH,
            repo_gate=REPO_GATE, gate_selftest=ST,
            bpm=dict(array_sha256=BPM_ARRAY_SHA256, file_sha256=BPM_FILE_SHA,
                     note='array hash = source of truth (rules v0.3)'),
            camb_cache_sha256=CAMB_CACHE_SHA, camb_array_sha256=CAMB_ARR_SHA,
            validation=dict(core_gate=VALID_GATE, bridge_battery=VALID_BRIDGE, camb=CAMB_VALID),
            completion=COMPLETION, r2_final=R2_FINAL,
            outputs=dict(raw_csv_sha256=_sha(CKPT),
                         extrapolated_csv_sha256=_sha(os.path.join(RUN_DIR, 't2b2_extrapolated.csv')),
                         figure_sha256=_sha(os.path.join(RUN_DIR, 't2b2_rhoq_map.png'))),
            impl_freeze=dict(alignment='body-y -> pix1134(N16,RING); psi: N_PSI=16 uniform (exact for l<=4)',
                             tol_neg_rel=1e-12, RHO_NUM=br.RHO_NUM, NOSIG_REL=br.NOSIG_REL,
                             DSMAX_meanbudget=tr.DSMAX, R2_exclude_families=list(tr.R2_EXCLUDE_FAMILIES),
                             g_eff='all normalizations absorbed into effective amplitude g_eff^2',
                             note_mc='Gaussian covariance-MC = implementation check only'))
json.dump(prov, open(os.path.join(RUN_DIR, 't2b2_provenance.json'), 'w'), indent=2, ensure_ascii=False)
print(json.dumps({k: prov[k] for k in ['run_mode', 'config_hash']}, ensure_ascii=False))
print('provenance saved:', os.path.join(RUN_DIR, 't2b2_provenance.json'))

## 実行手順（commit → push → 保存済みnotebookでrestart→Run all → production）
1. **先にコミット＆push**：本ノートブック（v0.6）＋`t2b2_core.py`（v0.3）＋`t2b2_bridge.py`
   （v1.2）＋`t2b2_run.py`（v1.2）を `mirror-topology`（正準URL）へ。規則v0.3はdocs/既存。
   **commit & push後はrepo内のnotebook sourceを一切変更しない**（NOTEBOOK_PATHはrepo内の
   tracked notebookへ自動解決されるため編集不要）。
2. **positive-path smoke**（Colab）：notebookの**スクラッチコピー**（Colabで「ドライブに
   コピーを保存」等）を開き，冒頭に`SUBSET='smoke'`の1セルを足して全実行。
   ※smokeではnotebook identityのassertは働かないためコピー編集で問題ありません。
   確認項目：`REPO_GATE.ok=True`・`nb_local_match=True`（自動解決先=repo内notebookとHEADの
   一致）・`origin_ok=True`・`pushed=True`・gate self-test OK・電池PASS・CAMB PASS・完了検査ok。
3. **production**：**同一commitのまま**，Driveのrepo clone内の**保存済み純正notebook**を開き，
   Runtime restart→上からRun all（SUBSET未設定＝'full'・セル追加や編集は一切しない）。
   ゲート通過後もセルを編集しない。
4. 実行後：`production_<hash>/`のraw/extrapolated CSV・provenance・図・電池出力を返送。
   final negペアが出た場合のV7 cross-checkは`tr.v7_crosscheck`をそのまま適用（凍結済み）。